# 02 - Preprocessing

This notebook turns the decisions from the EDA into a **leakage-free preprocessing pipeline** (`src/preprocessing.py`) and verifies that it behaves as intended.

**Why a pipeline?** If we impute or scale the whole dataset *before* splitting, information from the test set leaks into training. Here every statistic (medians, modes, means, standard deviations, category lists) is learned from the *training* data only - and, when the pipeline is used inside cross-validation, from each training fold only.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from src import preprocessing as pp

pd.set_option("display.max_columns", 40)

## 1. Train / test split

One stratified 80 / 20 split, fixed random seed, shared by every notebook and script. **The test set is not used for any modelling decision** - it is only used once, in notebook 04.

In [2]:
X_train, X_test, y_train, y_test = pp.get_split()

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Disease prevalence - train: {y_train.mean():.1%}   test: {y_test.mean():.1%}   (full data: {pp.load_data()['target'].mean():.1%})")

Train: (242, 13)   Test: (61, 13)
Disease prevalence - train: 45.9%   test: 45.9%   (full data: 45.9%)


## 2. Missing values

Only 6 values are missing (see notebook 01). Where do they fall after the split?

In [3]:
for split_name, part in [("train", X_train), ("test", X_test)]:
    m = part.isna().sum()
    m = m[m > 0]
    print(f"Missing values - {split_name}: " + (", ".join(f"{k}: {v}" for k, v in m.items()) if len(m) else "none"))

Missing values - train: ca: 1, thal: 1
Missing values - test: ca: 3, thal: 1


**Decision:** impute rather than delete, and do it *inside* the pipeline.

| Option | Rows lost | Verdict |
|---|---|---|
| `dropna()` | 6 of 303 (2 %) | Wastes scarce data and would need to be repeated consistently for new patients at prediction time |
| Impute inside the pipeline | 0 | **Chosen** - median for `ca` (a count), most-frequent for categorical codes; learned from training data only |

With 2 % missingness, the choice of imputation method is very unlikely to change conclusions.

## 3. Feature groups and transformations

In [4]:
groups = pd.DataFrame([
    ("Numeric (incl. ordinal `ca`)", ", ".join(pp.NUMERIC), "median imputation -> StandardScaler"),
    ("Binary (0/1)", ", ".join(pp.BINARY), "most-frequent imputation (values kept as 0/1)"),
    ("Nominal categories", ", ".join(pp.NOMINAL), "most-frequent imputation -> OneHotEncoder"),
], columns=["group", "features", "transformation"])
groups

,group,features,transformation
0,Numeric (incl. ordinal `ca`),"age, trestbps, chol, thalach, oldpeak, ca",median imputation -> StandardScaler
1,Binary (0/1),"sex, fbs, exang",most-frequent imputation (values kept as 0/1)
2,Nominal categories,"cp, restecg, slope, thal",most-frequent imputation -> OneHotEncoder


In [5]:
prep = pp.build_preprocessor()
print(prep)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='median')),
                                                 ('scale', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalach',
                                  'oldpeak', 'ca']),
                                ('bin',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['sex', 'fbs', 'exang']),
                                ('nom',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
            

## 4. Fit on the training set, transform both sets

In [6]:
prep.fit(X_train)
names = pp.pretty_feature_names(prep.get_feature_names_out())

Xt_train = pd.DataFrame(prep.transform(X_train), columns=names, index=X_train.index)
Xt_test = pd.DataFrame(prep.transform(X_test), columns=names, index=X_test.index)

print(f"Raw features: {X_train.shape[1]}  ->  model inputs after preprocessing: {Xt_train.shape[1]}")
Xt_train.head()

Raw features: 13  ->  model inputs after preprocessing: 22


,age,trestbps,chol,thalach,oldpeak,ca,sex,fbs,exang,cp_1,cp_2,cp_3,cp_4,restecg_0,restecg_1,restecg_2,slope_1,slope_2,slope_3,thal_3,thal_6,thal_7
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,-0.689715,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,-0.689715,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,0.445734,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,-0.689715,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,-0.689715,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


## 5. Verification

In [7]:
# 1) No missing values remain
print("NaNs after transform  -> train:", int(Xt_train.isna().sum().sum()), "  test:", int(Xt_test.isna().sum().sum()))

# 2) Numeric features are standardised using TRAIN statistics only
num_names = pp.NUMERIC
check = pd.DataFrame({
    "train mean": Xt_train[num_names].mean(),
    "train std": Xt_train[num_names].std(ddof=0),
    "test mean": Xt_test[num_names].mean(),
    "test std": Xt_test[num_names].std(ddof=0),
}).round(3)
check

NaNs after transform  -> train: 0   test: 0


,train mean,train std,test mean,test std
age,-0.0,1.0,-0.061,1.023
trestbps,-0.0,1.0,0.206,0.978
chol,0.0,1.0,-0.296,0.857
thalach,-0.0,1.0,-0.078,1.040
oldpeak,0.0,1.0,0.179,1.150
ca,0.0,1.0,0.315,1.235


In [8]:
# 3) What the imputers learned (from the training data)
num_imp = prep.named_transformers_["num"]["impute"]
nom_imp = prep.named_transformers_["nom"]["impute"]
print("Median used for numeric features:", dict(zip(pp.NUMERIC, num_imp.statistics_.round(2))))
print("Most-frequent value used for categorical features:", dict(zip(pp.NOMINAL, nom_imp.statistics_)))

# 4) One-hot columns created
print("\nOne-hot columns:", [n for n in names if n.split("_")[0] in pp.NOMINAL])

Median used for numeric features: {'age': np.float64(56.0), 'trestbps': np.float64(130.0), 'chol': np.float64(244.5), 'thalach': np.float64(153.5), 'oldpeak': np.float64(0.8), 'ca': np.float64(0.0)}
Most-frequent value used for categorical features: {'cp': np.float64(4.0), 'restecg': np.float64(0.0), 'slope': np.float64(1.0), 'thal': np.float64(3.0)}

One-hot columns: ['cp_1', 'cp_2', 'cp_3', 'cp_4', 'restecg_0', 'restecg_1', 'restecg_2', 'slope_1', 'slope_2', 'slope_3', 'thal_3', 'thal_6', 'thal_7']


**Verification results**

* No missing values remain after the transformation.
* The training set has mean 0 and standard deviation 1 for every scaled feature, while the test set does **not** (its values differ slightly) - proof that the scaler was fitted on the training data only and nothing from the test set leaked in.
* 13 raw features become 22 model inputs: 6 scaled numeric, 3 binary and 13 one-hot columns (`cp`: 4, `restecg`: 3, `slope`: 3, `thal`: 3).

## 6. Summary

The preprocessing lives in `src/preprocessing.py` (`build_preprocessor()`), and is the first step of every model `Pipeline` in `src/train.py`. Because it is part of the pipeline, cross-validation re-fits it inside every fold automatically.

**Next:** `03_model_training.ipynb` - four models, hyper-parameter tuning and nested cross-validation.